# IMPORT LIBRARY

In [1]:
!pip install gdown

In [2]:
import pandas as pd
import numpy as np
import tensorflow as tf
import gdown

from sklearn.preprocessing import LabelEncoder

# LOAD CLEANED DATASET

In [3]:
df = pd.read_csv('cleaned_sample_data.csv')

df.head()

,user_id,product_id,rating,timestamp
0,A1PSUH0U1FPQ6R,B002QXZPFE,4.0,2013-08-19
1,A23QSTB241NRF3,B0040HJOO2,1.0,2012-04-24
2,A1IU4JZFDZA9HJ,B004LRO7FW,5.0,2012-01-12
3,A1B2D4J8KF4DFN,B006DKEQL0,2.0,2013-05-10
4,A2SYLJAZO4SPA0,B00006G33N,4.0,2006-08-22


# LOAD MODEL

In [4]:
file_id = "1yBX-TqCxQQ8qPESF2HY8Fh6aFUJgrvCW"

url = f"https://drive.google.com/uc?id={file_id}"

gdown.download(
    url,
    "ncf_model.keras",
    quiet=False
)

model = tf.keras.models.load_model(
    'ncf_model.keras'
)

Downloading...
From (original): https://drive.google.com/uc?id=1yBX-TqCxQQ8qPESF2HY8Fh6aFUJgrvCW
From (redirected): https://drive.google.com/uc?id=1yBX-TqCxQQ8qPESF2HY8Fh6aFUJgrvCW&confirm=t&uuid=75f5b92c-7bad-4355-a30a-489f17652c54
To: /content/ncf_model.keras
100%|██████████| 31.5M/31.5M [00:00<00:00, 89.0MB/s]


# ENCODING USER DAN PRODUK

In [5]:
user_encoder = LabelEncoder()
product_encoder = LabelEncoder()

df['user_encoded'] = user_encoder.fit_transform(df['user_id'])

df['product_encoded'] = product_encoder.fit_transform(df['product_id'])

# MENAMPILKAN JUMLAH USER DAN PRODUK

In [6]:
num_users = df['user_encoded'].nunique()

num_products = df['product_encoded'].nunique()

print(f"Jumlah user: {num_users}")

print(f"Jumlah produk: {num_products}")

Jumlah user: 13345
Jumlah produk: 27447


# MEMBUAT MAPPING USER DAN PRODUK

In [7]:
user_id_mapping = dict(
    zip(
        df['user_encoded'],
        df['user_id']
    )
)

product_id_mapping = dict(
    zip(
        df['product_encoded'],
        df['product_id']
    )
)

# MEMILIH USER SAMPLE

In [8]:
sample_user_id = df['user_id'].iloc[0]

print(f"Sample User ID: {sample_user_id}")

Sample User ID: A1PSUH0U1FPQ6R


# ENCODE SAMPLE USER

In [9]:
sample_user_encoded = user_encoder.transform(
    [sample_user_id]
)[0]

print(f"Encoded User: {sample_user_encoded}")

Encoded User: 2540


# MENCARI PRODUK YANG SUDAH DIREVIEW USER

In [10]:
reviewed_products = df[
    df['user_id'] == sample_user_id
]['product_id'].tolist()

print(f"Jumlah produk yang sudah direview: {len(reviewed_products)}")

Jumlah produk yang sudah direview: 16


# MENCARI PRODUK YANG BELUM DIREVIEW

In [11]:
all_products = df['product_id'].unique()

unreviewed_products = [
    product
    for product in all_products
    if product not in reviewed_products
]

print(f"Jumlah produk yang belum direview: {len(unreviewed_products)}")

Jumlah produk yang belum direview: 27431


# ENCODE PRODUK YANG BELUM DIREVIEW

In [12]:
unreviewed_products_encoded = product_encoder.transform(
    unreviewed_products
)

# MEMBUAT INPUT MODEL

In [13]:
user_input = np.array(
    [sample_user_encoded] * len(unreviewed_products_encoded)
)

product_input = np.array(
    unreviewed_products_encoded
)

# MELAKUKAN PREDIKSI RATING

In [14]:
predicted_ratings = model.predict([
    user_input,
    product_input
])

858/858 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step


# MEMBUAT DATAFRAME HASIL PREDIKSI

In [15]:
recommendation_df = pd.DataFrame({
    'product_id': unreviewed_products,
    'predicted_rating': predicted_ratings.flatten()
})

recommendation_df.head()

,product_id,predicted_rating
0,B0040HJOO2,3.907328
1,B004LRO7FW,4.106995
2,B006DKEQL0,4.110240
3,B00006G33N,4.209672
4,B000FNC2IK,3.936626


# MENGURUTKAN REKOMENDASI

In [16]:
recommendation_df = recommendation_df.sort_values(
    by='predicted_rating',
    ascending=False
)

# MENAMPILKAN TOP 10 REKOMENDASI

In [17]:
top_10_recommendations = recommendation_df.head(10)

print("TOP 10 REKOMENDASI PRODUK")

top_10_recommendations

TOP 10 REKOMENDASI PRODUK


,product_id,predicted_rating
2594,B001TH7GSW,4.939638
3221,B00004Z5M1,4.826464
1350,B009FC3YJ8,4.819298
17286,B0064P5I1G,4.785554
11818,B000W09ZTK,4.781694
13080,B009APBY0G,4.772071
591,B00D5T3QK4,4.768299
6992,B0000ALLYO,4.753619
8008,B000SSLR5Q,4.745579
68,B004GF8TIK,4.743068


# MENAMPILKAN PRODUK YANG SUDAH DIREVIEW USER

In [18]:
print("PRODUK YANG SUDAH DIREVIEW USER:")

reviewed_df = pd.DataFrame({
    'reviewed_products': reviewed_products[:10]
})

reviewed_df

PRODUK YANG SUDAH DIREVIEW USER:


,reviewed_products
0,B002QXZPFE
1,B008GAMNBK
2,B000H6AY6M
3,B00I4Q9S32
4,B00AEGRGNO
5,B00385H7RS
6,B00ANJRTXE
7,B007KC1R3A
8,B007ZG32I4
9,B002CQU14A


# FUNGSI RECOMMENDATION SYSTEM

In [19]:
def recommend_products(user_id, top_n=10):

    # encode user
    user_encoded = user_encoder.transform([user_id])[0]

    # produk yang sudah direview
    reviewed_products = df[
        df['user_id'] == user_id
    ]['product_id'].tolist()

    # produk yang belum direview
    unreviewed_products = [
        product
        for product in all_products
        if product not in reviewed_products
    ]

    # encode produk
    unreviewed_products_encoded = product_encoder.transform(
        unreviewed_products
    )

    # input model
    user_input = np.array(
        [user_encoded] * len(unreviewed_products_encoded)
    )

    product_input = np.array(
        unreviewed_products_encoded
    )

    # prediksi
    predicted_ratings = model.predict([
        user_input,
        product_input
    ])

    # dataframe hasil prediksi
    recommendation_df = pd.DataFrame({
        'product_id': unreviewed_products,
        'predicted_rating': predicted_ratings.flatten()
    })

    # sorting
    recommendation_df = recommendation_df.sort_values(
        by='predicted_rating',
        ascending=False
    )

    return recommendation_df.head(top_n)

# TESTING FUNGSI REKOMENDASI

In [20]:
recommend_products(sample_user_id)

858/858 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step


,product_id,predicted_rating
2594,B001TH7GSW,4.939638
3221,B00004Z5M1,4.826464
1350,B009FC3YJ8,4.819298
17286,B0064P5I1G,4.785554
11818,B000W09ZTK,4.781694
13080,B009APBY0G,4.772071
591,B00D5T3QK4,4.768299
6992,B0000ALLYO,4.753619
8008,B000SSLR5Q,4.745579
68,B004GF8TIK,4.743068


# KESIMPULAN RECOMMENDATION SYSTEM:

1. Model berhasil memberikan rekomendasi produk
   berdasarkan pola interaksi user dan produk.

2. Recommendation system menggunakan Neural
   Collaborative Filtering (NCF).

3. Sistem dapat memprediksi rating produk yang
   belum pernah direview oleh user.

4. Produk dengan prediksi rating tertinggi
   direkomendasikan kepada user.